# Backtest ATR Trend Envelope Mod1 - USDCHF
Este notebook realiza a simulação do Expert Advisor `ATR_Trend_envelope_mod1.mq5` para o par USDCHF (Proxy: USDCHF=X) utilizando dados do Yahoo Finance.

### Regras Operacionais (Mod1):
1. **Compra**: Manutenção do suporte (Smin) por 2 candles + Rompimento da máxima no nível 23.6% de Fibonacci.
2. **Venda**: Manutenção da resistência (Smax) por 2 candles + Rompimento da máxima no nível 61.8% de Fibonacci.
3. **Gestão**: Stop Loss estrutural no envelope oposto e Trailing Stop seguindo a linha de tendência ativa.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Download de Dados Históricos
Baixando dados de 1 hora para USDCHF.

In [ ]:
data = yf.download('USDCHF=X', start='2025-03-14', end='2026-03-16', interval='1h')
df = data.copy()
df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
df.dropna(inplace=True)
df.head()

## 2. Implementação do Indicador ATR Trend Envelope Fibo
Calculando o ATR, as bandas do envelope e os níveis de Fibonacci solicitados.

In [ ]:
inpAtrPeriod = 14
inpDeviation = 1.5

df['TR'] = 0.0
for i in range(1, len(df)):
    df.loc[df.index[i], 'TR'] = max(df.iloc[i]['High'], df.iloc[i-1]['Close']) - min(df.iloc[i]['Low'], df.iloc[i-1]['Close'])

df['ATR'] = df['TR'].rolling(window=inpAtrPeriod).mean()

s_min = np.zeros(len(df))
s_max = np.zeros(len(df))
trend = np.zeros(len(df))

for i in range(1, len(df)):
    atr = df.iloc[i]['ATR']
    if pd.isna(atr): continue
    dev = atr * inpDeviation
    val_h, val_l, val_c = df.iloc[i]['High'], df.iloc[i]['Low'], df.iloc[i]['Close']
    s_max[i], s_min[i] = val_h + dev, val_l - dev
    prev_trend, prev_s_max, prev_s_min = trend[i-1], s_max[i-1], s_min[i-1]
    
    if val_c > prev_s_max and prev_s_max > 0: trend[i] = 1
    elif val_c < prev_s_min and prev_s_min > 0: trend[i] = -1
    else: trend[i] = prev_trend
        
    if trend[i] > 0 and s_min[i] < prev_s_min: s_min[i] = prev_s_min
    if trend[i] < 0 and s_max[i] > prev_s_max: s_max[i] = prev_s_max

df['Smin'], df['Smax'], df['Trend'] = s_min, s_max, trend
df['Fibo236'] = df['Smin'] + (df['Smax'] - df['Smin']) * 0.236
df['Fibo618'] = df['Smin'] + (df['Smax'] - df['Smin']) * 0.618

## 3. Simulação da Estratégia Mod1
Aplicando os filtros de manutenção de linha e cruzamentos Fibo.

In [ ]:
balance = 100000.0
risk_percent = 0.01
contract_size = 100000
history = []
positions = []

for i in range(2, len(df)):
    # Regra de Entrada
    if len(positions) == 0:
        # Compra: Trend UP + Suporte constante por 2 bars + High crosses Fibo 23.6
        if df.iloc[i-1]['Trend'] == 1 and df.iloc[i-2]['Trend'] == 1:
            if abs(df.iloc[i-1]['Smin'] - df.iloc[i-2]['Smin']) < 0.00001:
                if df.iloc[i-1]['High'] > df.iloc[i-1]['Fibo236'] and df.iloc[i-2]['High'] <= df.iloc[i-2]['Fibo236']:
                    entry = df.iloc[i]['Open']
                    sl = df.iloc[i-1]['Smax'] # Regra literal mod1
                    if sl >= entry: sl = df.iloc[i-1]['Smin'] - 0.0010 # Fallback de harmonização
                    positions.append({'dir': 1, 'entry': entry, 'sl': sl, 'lot': (balance*risk_percent)/(abs(entry-sl)*contract_size)})
        
        # Venda: Trend DOWN + Resistência constante por 2 bars + High crosses Fibo 61.8
        elif df.iloc[i-1]['Trend'] == -1 and df.iloc[i-2]['Trend'] == -1:
            if abs(df.iloc[i-1]['Smax'] - df.iloc[i-2]['Smax']) < 0.00001:
                if df.iloc[i-1]['High'] > df.iloc[i-1]['Fibo618'] and df.iloc[i-2]['High'] <= df.iloc[i-2]['Fibo618']:
                    entry = df.iloc[i]['Open']
                    sl = df.iloc[i-1]['Smin'] # Regra literal mod1
                    if sl <= entry: sl = df.iloc[i-1]['Smax'] + 0.0010 # Fallback de harmonização
                    positions.append({'dir': -1, 'entry': entry, 'sl': sl, 'lot': (balance*risk_percent)/(abs(entry-sl)*contract_size)})

    # Gestão de Posição
    for p in positions[:]:
        curr_h, curr_l, curr_c = df.iloc[i]['High'], df.iloc[i]['Low'], df.iloc[i]['Close']
        # SL Hit
        if (p['dir'] == 1 and curr_l <= p['sl']) or (p['dir'] == -1 and curr_h >= p['sl']):
            history.append((p['sl'] - p['entry']) * p['dir'] * p['lot'] * contract_size)
            positions.remove(p)
        else:
            # Trailing Stop Mod1: Segue a upLine (Smin) para compra ou dnLine (Smax) para venda
            if p['dir'] == 1:
                new_sl = df.iloc[i]['Smin']
                if new_sl > p['sl']: p['sl'] = new_sl
            else:
                new_sl = df.iloc[i]['Smax']
                if new_sl < p['sl'] or p['sl'] == 0: p['sl'] = new_sl
            
            # Fechamento por reversão de trend
            if df.iloc[i]['Trend'] != p['dir'] and df.iloc[i]['Trend'] != 0:
                history.append((curr_c - p['entry']) * p['dir'] * p['lot'] * contract_size)
                positions.remove(p)

## 4. Resultados e Métricas

In [ ]:
history = np.array(history)
if len(history) > 0:
    net_profit = np.sum(history)
    pf = abs(np.sum(history[history>0])/np.sum(history[history<=0])) if np.any(history<=0) else np.inf
    sharpe = (np.mean(history)/np.std(history))*np.sqrt(252) if np.std(history)>0 else 0
    
    cumulative = np.cumsum(history)
    balance_curve = 100000 + cumulative
    peak = 100000
    max_dd = 0
    for v in balance_curve:
        if v > peak: peak = v
        dd = (peak - v)/100000 * 100
        if dd > max_dd: max_dd = dd
        
    print(f"Lucro Líquido Total: {net_profit:.2f} USD")
    print(f"Fator de Lucro: {pf:.2f}")
    print(f"Índice Sharpe: {sharpe:.2f}")
    print(f"Drawdown Máximo: {max_dd:.2f}%")
    
    plt.figure(figsize=(12, 6))
    plt.plot(balance_curve, label='Equity', color='green')
    plt.axhline(100000, color='red', linestyle='--')
    plt.title('Evolução do Saldo USDCHF - Mod1 Strategy')
    plt.grid(True); plt.legend(); plt.show()
else:
    print("Nenhuma negociação executada no período.")